# Demo 14 - Stack counting (least frequency of occurrence)

**Pool:** Medium · **Visual:** long-tail frequency curve + rarest pairs

**The question:** which process launches are so rare that they are worth a look?

Count how often every parent-to-child process pair occurred across the whole estate, sort
from rarest upward, and read the bottom of the list. It is the oldest technique in threat
hunting and it still works, because malicious is almost always rare.

KQL can produce the counts. What it cannot give you is the long-tail curve that shows where
"rare" actually starts, or the ability to keep exploring the distribution without
re-querying.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `LOOKBACK_DAYS` - how much process history to stack.
- `MAX_FREQ` - what counts as "rare". The default treats a parent-child pair seen five times
  or fewer across the entire estate as rare.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 14
MAX_FREQ = 5          # "rare" = seen this many times or fewer across the whole estate

## 3. Stack every parent-child process pair

**Stack counting** is the oldest technique in threat hunting and it still works: count how
often every distinct thing happens, sort ascending, and read the bottom of the list. It is
also called least-frequency-of-occurrence, or LFO.

Here the "thing" is a process lineage - which program launched which other program. Names
are lowercased so `PowerShell.exe` and `powershell.exe` are counted as one.

A null parent is kept rather than dropped, and labelled `(no parent)`. An orphaned process
is a genuine hunting signal, and quietly discarding it would hide exactly what you came
looking for.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

proc = data_provider.read_table("DeviceProcessEvents", WORKSPACE)
proc = proc.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))

# Keep orphan processes (a null parent is a real hunting signal) but label them explicitly
# rather than rendering the string "None" in the chart.
stack = (proc.groupBy(
            F.coalesce(F.lower("InitiatingProcessFileName"), F.lit("(no parent)")).alias("parent"),
            F.coalesce(F.lower("FileName"), F.lit("(no child)")).alias("child"))
          .agg(F.count("*").alias("freq"),
               F.countDistinct("DeviceName").alias("devices"))
          .orderBy("freq"))     # rarest first = LFO
pdf = stack.toPandas()
print("distinct parent->child pairs:", len(pdf))

## 4. Read the long tail

Left chart: every pair sorted from rarest to most common, on a log scale. This is the shape
you are hunting - a huge flat body on the right, which is your normal, falling away into a
thin tail on the left. The tail is where you look.

Right chart: the fifteen rarest lineages, labelled.

**What to look for:** `winword.exe -> powershell.exe`, or `outlook.exe -> cmd.exe`. Something
ordinary launching something scripting-capable, exactly once, on exactly one machine.

Rare is not the same as malicious - most of the tail will be installers and one-off admin
work - but malicious is almost always rare, which is what makes the tail worth reading.

In [ ]:
if pdf.empty:
    print(f"No process events in the last {LOOKBACK_DAYS} days - raise LOOKBACK_DAYS.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].plot(range(len(pdf)), pdf["freq"].values, color="#2c3e50")
    ax[0].set_yscale("log"); ax[0].set_title("Frequency distribution (log) - hunt the left edge")
    ax[0].set_xlabel("parent->child pair (sorted rarest -> common)"); ax[0].set_ylabel("occurrences")

    rare = pdf[pdf["freq"] <= MAX_FREQ].head(15).iloc[::-1]
    labels = [f'{i+1}. {r.parent} -> {r.child}'[:46] for i, r in enumerate(rare.itertuples())]
    ax[1].barh(labels, rare["freq"].values, color="#c0392b")
    ax[1].set_title(f"Rarest process lineages (freq <= {MAX_FREQ})"); ax[1].set_xlabel("occurrences")
    plt.tight_layout(); plt.show()

pdf[pdf["freq"] <= MAX_FREQ].head(25)

## Why this is a notebook hunt, not a KQL query

Stack counting is a *distribution* hunt: sort by rarity and read the tail. Pandas makes the whole distribution explorable and plottable in one shot; in KQL you'd `summarize count()` and squint at rows without the long-tail curve that guides the eye.